<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 12 · ZeroMQ and Real-Time Market Data Sandboxes

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook explores the structure of the ZeroMQ tick server and its clients without starting long-running processes.

- Inspect the instrument configuration used for synthetic ticks.
- Simulate a short sequence of random-walk prices in notebook space.
- Understand the message format that streaming clients subscribe to.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})

PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
STREAM_DIR = CODE_DIR / "streaming"  # streaming utilities
if str(STREAM_DIR) not in sys.path:  # ensure streaming modules are importable
    sys.path.insert(0, str(STREAM_DIR))

from tick_server import InstrumentConfig  # configuration for synthetic ticks


### 1. Instrument Configuration

The tick server treats each symbol as a tiny configuration object. We mirror the defaults here and store them in a DataFrame for quick inspection.

In [ ]:
instruments = {
    "SPY": InstrumentConfig("SPY", start_price=500.0, vol=0.0005),
    "GLD": InstrumentConfig("GLD", start_price=200.0, vol=0.0007),
    "EURUSD": InstrumentConfig("EURUSD", start_price=1.10, vol=0.0003),
}

pd.DataFrame(
    {
        "symbol": [cfg.symbol for cfg in instruments.values()],
        "start_price": [cfg.start_price for cfg in instruments.values()],
        "vol": [cfg.vol for cfg in instruments.values()],
    }
).set_index("symbol")


### 2. Local Tick Simulation (Without ZeroMQ)

To understand the random-walk dynamics used in the server, we simulate a short sequence of ticks for one symbol directly in the notebook.

In [ ]:
rng = np.random.default_rng(seed=12)
cfg = instruments["SPY"]

n_ticks = 200
prices = np.empty(n_ticks)
prices[0] = cfg.start_price

for t in range(1, n_ticks):
    shock = rng.normal(loc=0.0, scale=cfg.vol)
    prices[t] = prices[t - 1] * (1.0 + shock)

fig, ax = plt.subplots(figsize=(7.0, 3.5))
ax.plot(prices)
ax.set_xlabel("tick index")
ax.set_ylabel("synthetic price")
ax.set_title("SPY-style synthetic ticks (local simulation)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### 3. Message Format

Tick messages published by the server follow the pattern `"SYMBOL ISO_TIMESTAMP PRICE"`. The next cell shows how such a string is assembled for one of our simulated prices.

In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc).isoformat()
example_price = float(prices[-1])
msg = f"{cfg.symbol} {now} {example_price:.5f}"
msg


### Takeaways

- The ZeroMQ sandbox uses very simple geometric random walks and human-readable message strings.
- You can prototype strategies against locally simulated ticks before connecting to the actual server process.
- Client scripts such as `client_recorder.py` and `client_event_backtester.py` simply parse the message format demonstrated here and feed events into familiar backtesting components.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">